In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import colorir as cl
import numpy as np
from pathlib import Path
import analyses.calculate_adh as adh
from analyses import io
import analyses.phylogeny as phy
from tqdm import tqdm

In [ ]:
celldf = (
    io
        .read_celldfs("../runs/evo/", levels=["replica"])
        .with_columns(
            replica=pl.col("replica").cast(pl.UInt32)
        )
        .sort(["replica", "wtime"])
)
celldf

In [ ]:
selectors = ["ligands", "receptors"]
gammadf = pl.DataFrame()
groupped = (
    celldf
        .select(selectors + ["replica", "wtime"])
        .filter(pl.col("wtime") % 2e6 == 0)  # Values only change between seasons of 2e6 steps
        .group_by(["replica", "wtime"])
)
for (replica, time), group in tqdm(groupped):
    sel_group = group.select(selectors)
    lig_recep = sel_group.select(selectors).join(sel_group, how="cross").to_numpy().T
    gammas = adh.calculate_gamma(
        20, 
        adh.cell_contact_energy(
            *lig_recep,
            20,
            32,
            8
        )
    )
    summary = (
        pl.DataFrame(gammas)
            .describe()
            .with_columns(replica=replica, wtime=time)
            .pivot(on="statistic", values="column_0")
    ).with_columns(sample=np.expand_dims(np.random.choice(gammas, 100), 0))
    gammadf = pl.concat([
        gammadf, 
        summary,
    ])
gammadf
    

In [ ]:
fig = go.Figure()
colors = cl.StackPalette.load("carnival").resize(gammadf["replica"].n_unique(), repeat=True)
for (replica,), group in gammadf.group_by("replica"):
    color = colors[7]
    line = go.Scatter(
        x=group["wtime"],
        y=group["mean"],
        mode="markers",
        line_color=color,
        opacity=0.1,
        legendgroup=replica,
        name=replica,
        marker_size=8,
        showlegend=False
    )
    fill = go.Scatter(
        x=pl.concat([group["wtime"], group["wtime"][::-1]]),
        y=pl.concat([group["25%"], group["75%"][::-1]]),
        fill="toself",
        line_color="rgba(0, 0, 0, 0)",
        fillcolor=color,
        opacity=0.2,
        legendgroup=replica
    )
    fig.add_trace(line)
    # fig.add_trace(fill)

gmeans = gammadf.group_by("wtime").agg(
    mean=pl.col("mean").median(),
    std=pl.col("mean").std(),
).sort("wtime")
fig.add_trace(
    go.Scatter(
        x=gmeans["wtime"],
        y=gmeans["mean"],
        line_color="#000000",
        line_dash="dash",
        name="median"
    )
)
# fig.add_trace(
#     go.Scatter(
#         x=pl.concat([gmeans["wtime"], gmeans["wtime"][::-1]]),
#         y=pl.concat([gmeans["mean"] + 1.5 * gmeans["std"], gmeans["mean"][::-1] - 1.5 * gmeans["std"][::-1]]),
#         fill="toself",
#         line_color="rgba(0, 0, 0, 0)",
#         opacity=0.4
#     )
# )
fig.update_layout(template="plotly_white")

In [ ]:
sampledf = gammadf.filter(pl.col("wtime") > 150e6).group_by("replica").agg(sample=pl.concat_arr(pl.col("sample")))
sampledf

In [ ]:
ancs = (
    celldf
        .filter(
            (pl.col("wtime") - 1e5) % 2e6 == 0,  # Only changes on time after season change
            replica=0)
        .select(["index", "ancestor", "wtime"])
        .sort("wtime")
)
ancdf = ancs.filter(wtime=ancs["wtime"].first()).select("index")
for (time,), group in ancs.group_by("wtime", maintain_order=True):
    if time == ancs["wtime"].first():
        continue
    ancdf = (
        group
            .join(
                ancdf,
                left_on="ancestor", 
                right_on="index",
                how="full",  # Change to left to exclude extinct lineages (same as dropping nulls from the matrix afterwards)
                coalesce=False
            )
            .select(["index", "index_right", pl.col(r"^\d*$")])
            .rename({
                "index_right": f"{time - int(2e6 + 1e5)}"
            })
    )
ancdf = ancdf.rename({"index": str(time - int(1e5))})
ancdf



In [ ]:
ancdf = phy.matrix(celldf.filter(replica=0), 2e6)
ancdf

In [ ]:
ancmelt = (
    ancdf
        .with_columns(lineage=np.arange(ancdf.height), og=ancdf["0"])
        .unpivot(index=["lineage", "og"], value_name="index", variable_name="wtime")
        .with_columns(pl.col("wtime").cast(int))
        .sort(["lineage", "wtime"])
)
ancmelt


In [ ]:
px.line(ancmelt, x="index", y="wtime", color="lineage").update_traces(visible="legendonly")

In [ ]:
descdf = ancmelt.group_by(["og", "wtime"]).agg(pl.col("index").drop_nulls().n_unique()).sort("og", "wtime")
descdf

In [ ]:
px.bar(
    descdf,
    x="wtime", 
    y="index", 
    color="og",
    barmode="stack"
).update_traces(
    marker_line_color="rgba(0, 0, 0, 0)"
).update_layout(
    bargap=0
)